# Appendix C — Calibrating GMS Decision Thresholds

*The chapter says calibrate it. This appendix shows how.*

In [ ]:
import sys, torch
if not torch.cuda.is_available():
    print(
        "This notebook requires a GPU (CUDA) — no GPU detected.\n"
        "Please re-run on a machine with a CUDA-capable GPU."
    )
    sys.exit(0)

## The thresholds that matter

| Threshold | What it controls | Default |
|---|---|---|
| `theta_plausibility` | Plausibility-gate cut on `score_triple` | 1.5 |
| `tau_contra` | Contradiction cut on `tension_energy` | ~1.7 |
| `numeric_tolerance` | NUMERIC-claim tolerance vs ENM | 0.01 |
| `holonomy_threshold` | Multi-hop escalation threshold | 0.5 |
| `epsilon_phase` | Inequality slack on phase encoder | 0.05 rad |

We walk through `theta_plausibility` against a labeled cohort, using the DOE machinery from Chapter 11.

In [ ]:
import torch
from dataclasses import dataclass
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore
from forgeloop.agents.evaluation import balanced_design, coverage_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code'))
             if (c / 'data' / 'gms_banking_store').exists()), Path('.'))
config = DocGMSConfig(store_path=str(ROOT / 'data' / 'gms_banking_store'))
store = GMSExpertStore(config, device=DEVICE)
store.load()

## A labeled cohort

Calibration needs labeled positives and negatives. The cohort below uses the workflow steps from the banking policy; positive examples are real workflow edges, negative examples are nonsense edges. Production cohorts are sampled from labeled production traces.

In [ ]:
@dataclass
class LabeledEdge:
    context: str
    relation: str
    tail: str
    label: str  # 'admit' or 'block'

# The real workflow-authorization edges (has_enables). Legal transitions are
# admits; skips and reversals are blocks. This is the cohort calibrate_gms_
# thresholds.py sweeps, so the walkthrough reproduces the shipped operating point.
cohort = [
    LabeledEdge('start',           'has_enables', 'classify',        'admit'),
    LabeledEdge('classify',        'has_enables', 'extract',         'admit'),
    LabeledEdge('extract',         'has_enables', 'search_policy',   'admit'),
    LabeledEdge('search_policy',   'has_enables', 'flag_regulatory', 'admit'),
    LabeledEdge('flag_regulatory', 'has_enables', 'draft_response',  'admit'),
    LabeledEdge('flag_regulatory', 'has_enables', 'escalate',        'admit'),
    LabeledEdge('draft_response',  'has_enables', 'escalate',        'admit'),
    LabeledEdge('classify',        'has_enables', 'search_policy',   'block'),  # skips extract
    LabeledEdge('classify',        'has_enables', 'draft_response',  'block'),  # skips three
    LabeledEdge('extract',         'has_enables', 'flag_regulatory', 'block'),  # skips search
    LabeledEdge('search_policy',   'has_enables', 'draft_response',  'block'),  # skips flag
    LabeledEdge('draft_response',  'has_enables', 'classify',        'block'),  # reversal
]
n_admit = sum(1 for c in cohort if c.label == 'admit')
n_block = sum(1 for c in cohort if c.label == 'block')
print(f'cohort: {len(cohort)} edges ({n_admit} admit, {n_block} block)')


## The sweep

For each candidate threshold value, score each cohort edge through `store.score_triple` and count false admits (block-labeled edges passing) and false rejects (admit-labeled edges failing). The threshold that minimizes both is the operating point.

In [ ]:
thetas = [0.15, 0.25, 0.35, 0.45, 0.55, 0.75]
rows = []
for theta in thetas:
    fa, fr, tp, tn = 0, 0, 0, 0
    for edge in cohort:
        s = store.score_triple(edge.context, edge.relation, edge.tail)
        if s is None:
            continue  # on_missing path; treat as no decision
        admitted = s <= theta
        if admitted and edge.label == 'block': fa += 1
        elif not admitted and edge.label == 'admit': fr += 1
        elif admitted and edge.label == 'admit': tp += 1
        elif not admitted and edge.label == 'block': tn += 1
    rows.append({'theta': theta, 'false_admits': fa, 'false_rejects': fr,
                 'accuracy': (tp + tn) / max(tp+tn+fa+fr, 1)})
for r in rows:
    print(f'  theta={r["theta"]:.2f}  FA={r["false_admits"]}  FR={r["false_rejects"]}  acc={r["accuracy"]:.2f}')


## Pick an operating point

The cost of an error is asymmetric:

- **False admit** — an implausible call goes through. Downstream gates may catch it.
- **False reject** — a legitimate call is denied. The agent has to replan or escalate.

In a regulated setting the cost of a false admit dominates, so we do not maximize accuracy. We take the threshold that admits none, then minimize the false rejects among those, breaking a tie toward the stricter (lower) threshold.

In [ ]:
# Regulated regime: admit no false admit, then minimize false rejects (tie -> lower
# theta). Not max-accuracy: the two errors do not cost the same.
zero_false_admit = [r for r in rows if r['false_admits'] == 0]
best = min(zero_false_admit or rows,
           key=lambda r: (r['false_admits'], r['false_rejects'], r['theta']))
print(f'best operating point: theta = {best["theta"]:.2f}  acc={best["accuracy"]:.2f}  FA={best["false_admits"]}  FR={best["false_rejects"]}')

## A DOE design over multiple thresholds

When more than one threshold is in play, the balanced design from Chapter 11 covers the joint factor space without running the full grid.

In [ ]:
factors = {
    'theta_plausibility': [0.25, 0.35, 0.45, 0.55],
    'tau_contra':         [0.9, 1.1, 1.3],
    'numeric_tolerance':  [0.005, 0.01, 0.02],
    'holonomy_threshold': [0.3, 0.5, 0.7],
}
design = balanced_design(factors, num_cases=12, seed=42)
print('design:')
for row in design[:4]:
    print(f'  {row}')
print('...')
print('coverage:')
for fname, counts in coverage_report(design, factors).items():
    print(f'  {fname}: {counts}')

## When to recalibrate

Recalibrate when the store has been re-trained, the drift monitor reports PSI above its threshold, the gate's false-reject rate creeps up, or domain rules change.

## Anti-patterns flagged here

- Calibrating on a cohort produced by the same agent.
- One-shot calibration. The substrate drifts; bundles expire.
- Choosing thresholds by eye on a precision-recall curve. Pick on the cost-weighted frontier.

In [ ]:
# Self-check: the operating point obeys the regulated rule (minimal false admits)
# and the DOE design is balanced.
assert best['false_admits'] == min(r['false_admits'] for r in rows), \
    'operating point must minimize false admits, not accuracy'
covered = coverage_report(design, factors)
for fname, counts in covered.items():
    assert max(counts.values()) - min(counts.values()) <= 1, f'{fname} not balanced'
print('OK')